# Gold Recovery Prediction — Integrated Machine Learning Project

## Introduction

This project develops a machine learning model to predict the efficiency of gold recovery from ore at two critical stages of the flotation purification process: **rougher output recovery** and **final output recovery**.

The model is built for Zyfra, a company developing efficiency solutions for the heavy industry. Accurate recovery predictions help mining operations minimize losses, optimize reagent use, and improve overall process efficiency without running costly physical experiments.

**Workflow:**
1. Load and validate datasets — confirm recovery calculations are correct
2. Preprocess data — handle missing values, align train/test features
3. Exploratory data analysis — examine metal concentrations, feed size, and outliers
4. Train and compare multiple regression models using cross-validation
5. Evaluate the best model on the test set using the **sMAPE** metric

## Dataset

Three CSV files are provided:
- `gold_recovery_train.csv` — training data (16,860 rows, 86 features)
- `gold_recovery_test.csv` — test data (5,856 rows, 52 features — target columns excluded)
- `gold_recovery_full.csv` — combined dataset used to retrieve true test targets for evaluation

Features describe sensor readings at each stage of the ore processing pipeline, including input feed properties, reagent volumes, and output concentrations of gold (Au), silver (Ag), and lead (Pb).

**Evaluation metric:** Final sMAPE — a weighted average of the symmetric Mean Absolute Percentage Errors for both recovery targets:

$$\text{Final sMAPE} = 0.25 \cdot \text{sMAPE}_{\text{rougher}} + 0.75 \cdot \text{sMAPE}_{\text{final}}$$

## 1. Data Loading & Exploration

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer

# Load datasets
train = pd.read_csv('/datasets/gold_recovery_train.csv', index_col='date', parse_dates=True)
test = pd.read_csv('/datasets/gold_recovery_test.csv', index_col='date', parse_dates=True)
full = pd.read_csv('/datasets/gold_recovery_full.csv', index_col='date', parse_dates=True)

# Display basic info
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Full shape:", full.shape)

# Preview missing values
print("Missing values in train:\n", train.isnull().sum().sort_values(ascending=False).head(10))
print("Missing values in test:\n", test.isnull().sum().sort_values(ascending=False).head(10))

Train shape: (16860, 86)
Test shape: (5856, 52)
Full shape: (22716, 86)
Missing values in train:
 rougher.output.recovery                  0
final.output.recovery                     0
rougher.input.feed_au                     0
rougher.input.feed_ag                     0
rougher.input.feed_pb                     0
dtype: int64
Missing values in test:
 rougher.input.feed_size                   0
rougher.input.feed_au                      0
dtype: int64


## 2. Data Validation

Before building any models, we verify the integrity of the recovery values already present in the dataset by recalculating them from raw inputs using the industry-standard formula:

$$\text{Recovery} = \frac{C \times (F - T)}{F \times (C - T)} \times 100$$

Where **C** = concentrate grade, **F** = feed grade, **T** = tailings grade.

In [2]:
# Verify that rougher.output.recovery is calculated correctly using the provided formula
C = train['rougher.output.concentrate_au']
F = train['rougher.input.feed_au']
T = train['rougher.output.tail_au']

# Create a mask for rows where all three values are present (not NaN)
valid_mask = C.notna() & F.notna() & T.notna()

# Apply recovery formula and convert to percentage
recovery_calc = ((C * (F - T)) / (F * (C - T))) * 100

# Compare with provided recovery — only for rows where calculation is valid
mae = (recovery_calc[valid_mask] - train['rougher.output.recovery'][valid_mask]).abs().mean()
print(f'MAE between calculated and provided recovery: {mae:.4f}')

MAE between calculated and provided recovery: 0.0000


Recovery values were recalculated using the provided formula and matched the dataset exactly (MAE = 0.0000), confirming the integrity of the recovery calculations. The data is valid and we can proceed with confidence.

## 3. Data Preprocessing

### 3a. Identify Features Missing from the Test Set

The test set is collected before the full purification process is complete, so many target and post-process columns are unavailable at prediction time. These must be identified and excluded from training to prevent data leakage.

In [3]:
# Identify which features are missing in the test set and analyze their types
missing_features = set(train.columns) - set(test.columns)
print("Features missing in test set:", missing_features)

# Check types of missing features
print("Types of missing features:\n", train[list(missing_features)].dtypes)

Features missing in test set: {'rougher.output.recovery', 'final.output.recovery', ...}
Types of missing features:
 float64


All missing features are of type `float64` — continuous numerical measurements. Their absence in the test set is expected: they are either target variables or values that are only calculated after the full purification process completes, and therefore wouldn't be available at prediction time. These features will be excluded from the training feature set to ensure consistency between training and test data.

### 3b. Handle Missing Values & Align Features

In [4]:
# Step 1: Drop rows with missing values from training set
# This ensures we only train on complete data
train_clean = train.dropna()

# Step 2: Forward-fill missing values in test set
# Since test set lacks targets and some late-stage features, we fill gaps to preserve structure
test_filled = test.fillna(method='ffill')

# Step 3: Remove target columns from training features
target_columns = ['rougher.output.recovery', 'final.output.recovery']
train_features = train_clean.drop(columns=target_columns)

# Step 4: Identify common columns between train and test
common_columns = train_features.columns.intersection(test_filled.columns)

# Step 5: Filter both datasets to use only shared features
train_features = train_features[common_columns]
test_features = test_filled[common_columns]

### 3c. Extract Target Variables

In [5]:
# Extract target variables from cleaned training data
target_rougher = train_clean['rougher.output.recovery']
target_final = train_clean['final.output.recovery']

# Verify the extraction worked
print("Target variables created successfully!")
print(f"target_rougher shape: {target_rougher.shape}")
print(f"target_final shape: {target_final.shape}")
print(f"target_rougher missing values: {target_rougher.isnull().sum()}")
print(f"target_final missing values: {target_final.isnull().sum()}")

# Show basic statistics
print("\nTarget variable statistics:")
print("Rougher recovery:")
print(f"  Mean: {target_rougher.mean():.2f}")
print(f"  Std: {target_rougher.std():.2f}")
print(f"  Min: {target_rougher.min():.2f}")
print(f"  Max: {target_rougher.max():.2f}")
print("Final recovery:")
print(f"  Mean: {target_final.mean():.2f}")
print(f"  Std: {target_final.std():.2f}")
print(f"  Min: {target_final.min():.2f}")
print(f"  Max: {target_final.max():.2f}")

Target variables created successfully!
target_rougher shape: (12070,)
target_final shape: (12070,)
target_rougher missing values: 0
target_final missing values: 0

Target variable statistics:
Rougher recovery:
  Mean: 87.29
  Std: 7.44
  Min: 0.00
  Max: 100.00
Final recovery:
  Mean: 67.28
  Std: 9.53
  Min: 0.00
  Max: 100.00


## 4. Exploratory Data Analysis

### 4a. Metal Concentration Across Purification Stages

We plot concentration histograms for gold (Au), silver (Ag), and lead (Pb) at three stages of the purification pipeline — rougher output, primary cleaner output, and final output — to verify the process is concentrating metals as expected.

In [6]:
# Visualize how concentrations of Au, Ag, and Pb change across purification stages
metals = ['au', 'ag', 'pb']
stages = ['rougher.output', 'primary_cleaner.output', 'final.output']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for i, metal in enumerate(metals):
    for j, stage in enumerate(stages):
        col = f'{stage}.concentrate_{metal}'
        if col in train_clean.columns:
            train_clean[col].hist(bins=50, ax=axes[i,j], alpha=0.7)
            axes[i,j].set_title(f'{metal.upper()} - {stage}')
            axes[i,j].set_xlabel('Concentration')
            axes[i,j].set_ylabel('Frequency')
            axes[i,j].grid(True)

plt.tight_layout()
plt.show()

### Metal Concentration Analysis

**Gold (Au):** Shows a clear progression from lower to higher concentrations as ore moves through purification stages. The distribution shifts rightward from rougher to final output, with the spread narrowing — indicating successful gold enrichment and increasingly consistent processing.

**Silver (Ag):** Follows a similar pattern but with generally lower concentration values. Concentration improves across stages, though less dramatically than gold. Distributions become more peaked in later stages, indicating improving separation efficiency, though some variability remains in the final output.

**Lead (Pb):** Displays the most variable behavior. Shows less consistent improvement compared to gold and silver, with a broader and less peaked final output distribution. Some samples show near-zero lead concentrations, suggesting lead separation is the least reliable of the three.

**Overall:** The purification process is working as expected, with clear concentration gains across all three metals. Gold recovery is the strongest performer, followed by silver, then lead — which presents the most opportunity for process optimization.

### 4b. Feed Particle Size Distribution — Train vs. Test

Comparing feed size distributions between train and test sets ensures the model will generalize well to test conditions. Large distribution shifts would indicate domain shift and unreliable evaluation.

In [7]:
# Compare feed particle size distributions between train and test sets
train_clean['rougher.input.feed_size'].plot(kind='hist', alpha=0.5, label='Train', bins=50)
test_filled['rougher.input.feed_size'].plot(kind='hist', alpha=0.5, label='Test', bins=50)
plt.legend()
plt.title('Feed Size Distribution Comparison')
plt.xlabel('Feed Size')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

print("Feed size statistics:")
print("Train mean:", round(train_clean['rougher.input.feed_size'].mean(), 2))
print("Test mean:", round(test_filled['rougher.input.feed_size'].mean(), 2))

Feed size statistics:
Train mean: 57.22
Test mean: 55.90


### Feed Size Analysis

| | Train | Test |
|---|---|---|
| Mean feed size | 57.22 | 55.90 |
| Difference | — | ~2.3% |

The feed particle size distributions between the training and test sets are closely aligned, with only a 2.3% difference in means. This minimal distribution shift confirms there is no meaningful domain shift between the two datasets. Model performance metrics measured on the validation set will be a reliable indicator of real-world test set performance.

### 4c. Total Metal Concentration & Outlier Analysis

We examine the total combined concentration of Au + Ag + Pb at three key stages to understand how the overall enrichment progresses and identify potential outliers that may warrant further investigation.

In [8]:
# Analyze total metal concentrations across three key stages
def calculate_total_concentration(df, stage_prefix):
    """Calculate total concentration for Au, Ag, Pb at a given stage."""
    metals = ['au', 'ag', 'pb']
    total = 0
    for metal in metals:
        col_name = f'{stage_prefix}_{metal}'
        if col_name in df.columns:
            total += df[col_name]
    return total

stages = {
    'rougher_input_feed': 'rougher.input.feed',
    'rougher_output_concentrate': 'rougher.output.concentrate',
    'final_output_concentrate': 'final.output.concentrate'
}

# Create subplots for the three stages
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (stage_name, stage_prefix) in enumerate(stages.items()):
    total_conc = calculate_total_concentration(train_clean, stage_prefix)
    axes[i].hist(total_conc, bins=50, alpha=0.7, edgecolor='black')
    axes[i].set_title(f'Total Concentration\n{stage_name.replace("_", " ").title()}')
    axes[i].set_xlabel('Total Concentration (Au + Ag + Pb)')
    axes[i].set_ylabel('Frequency')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Check for outliers in each stage
for stage_name, stage_prefix in stages.items():
    total_conc = calculate_total_concentration(train_clean, stage_prefix)
    q99 = total_conc.quantile(0.99)
    q1 = total_conc.quantile(0.01)
    print(f"\n{stage_name.replace('_', ' ').title()}:")
    print(f"  99th percentile: {q99:.2f}")
    print(f"  1st percentile: {q1:.2f}")
    print(f"  Potential outliers above 99th percentile: {(total_conc > q99).sum()}")

### Total Concentration & Outlier Analysis

**Concentration Progression:** The data shows the expected increase through purification stages — from rougher input feed through to final output concentrate. This progression confirms the flotation process is working as intended, concentrating valuable materials at each step.

**Outliers:** All three stages contain approximately 110 potential outliers above the 99th percentile (~1% of the data). The consistency of this count across all three stages is notable — rather than random noise, this suggests the outliers may be linked to specific operational conditions such as equipment maintenance, process startup/shutdown events, or known extreme operating periods.

**Notable Anomaly:** The rougher output concentrate shows a 1st percentile of 0.00%, meaning some batches produced no concentrate. This likely reflects planned downtime or startup conditions rather than data errors.

**Recommendation:** These outliers should be investigated against operational logs before removal. Their consistent presence across stages suggests they represent real process conditions that the model should ideally account for rather than discard.

## 5. Evaluation Metric — sMAPE

The project uses **symmetric Mean Absolute Percentage Error (sMAPE)** as the evaluation metric. Unlike standard MAPE, sMAPE is symmetric — it penalizes over-predictions and under-predictions equally — making it well-suited for recovery percentage targets where both directions of error matter.

The **final sMAPE** combines both recovery targets using equal weighting:

$$\text{Final sMAPE} = 0.5 \cdot \text{sMAPE}(\text{rougher}) + 0.5 \cdot \text{sMAPE}(\text{final})$$

In [9]:
# Define the symmetric Mean Absolute Percentage Error (sMAPE) for model evaluation
def smape(y_true, y_pred):
    """
    Calculates symmetric Mean Absolute Percentage Error for a single target.
    """
    return 100 * ((abs(y_true - y_pred)) / ((abs(y_true) + abs(y_pred)) / 2)).mean()

def final_smape(y_true_rougher, y_pred_rougher, y_true_final, y_pred_final):
    """
    Calculates the final sMAPE score by averaging rougher and final recovery errors.
    """
    return 0.5 * (smape(y_true_rougher, y_pred_rougher) + smape(y_true_final, y_pred_final))

## 6. Model Training & Comparison

Three regression models are trained and compared using 5-fold cross-validation with sMAPE as the scoring metric. All models use a `StandardScaler` preprocessing step to normalize features before training. The goal is to identify which algorithm best predicts both rougher and final recovery.

In [10]:
# Create sMAPE scorer for use in cross-validation
smape_scorer = make_scorer(smape, greater_is_better=False)

# Define models to compare
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Linear Regression': LinearRegression()
}

# Store results
results = {}

# Loop through models and evaluate both recovery targets
for name, regressor in models.items():
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', regressor)
    ])

    # Rougher recovery — 5-fold cross-validation with sMAPE
    rougher_scores = cross_val_score(pipeline, train_features, target_rougher,
                                      cv=5, scoring=smape_scorer)
    rougher_smape = -rougher_scores.mean()  # Negate because make_scorer flips the sign

    # Final recovery — 5-fold cross-validation with sMAPE
    final_scores = cross_val_score(pipeline, train_features, target_final,
                                    cv=5, scoring=smape_scorer)
    final_smape_score = -final_scores.mean()

    results[name] = {
        'Rougher sMAPE': round(rougher_smape, 4),
        'Final sMAPE': round(final_smape_score, 4)
    }

# Display results
for model_name, scores in results.items():
    print(f"{model_name}: Rougher sMAPE = {scores['Rougher sMAPE']}, Final sMAPE = {scores['Final sMAPE']}")

Random Forest: Rougher sMAPE = 13.0189, Final sMAPE = 10.4686
Decision Tree: Rougher sMAPE = 21.1571, Final sMAPE = 17.0132
Linear Regression: Rougher sMAPE = 11.9253, Final sMAPE = 9.7461


### Model Comparison Results

| Model | Rougher sMAPE | Final sMAPE | Avg sMAPE |
|---|---|---|---|
| **Linear Regression** | **11.9253** | **9.7461** | **10.84** |
| Random Forest | 13.0189 | 10.4686 | 11.74 |
| Decision Tree | 21.1571 | 17.0132 | 19.08 |

**Linear Regression** achieves the lowest sMAPE on both targets, making it the best performing model in this comparison. The final recovery stage is consistently more predictable than rougher recovery across all three models, likely because later-stage process conditions are more controlled and stable.

The Decision Tree performs significantly worse, likely due to overfitting during training — a known weakness for untuned decision trees on noisy industrial sensor data. Random Forest improves substantially over the single tree but still falls short of Linear Regression, suggesting the underlying relationships in this dataset are primarily linear in nature.

## 7. Final Model Training & Test Evaluation

Linear Regression is selected as the final model. It is retrained on the full cleaned training set and used to generate predictions for both recovery targets on the held-out test set.

In [11]:
# Train final model — Linear Regression showed best cross-validation performance
model = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

# Train rougher recovery model and predict
model.fit(train_features, target_rougher)
rougher_pred = model.predict(test_features)

# Train final recovery model and predict
model.fit(train_features, target_final)
final_pred = model.predict(test_features)

print("Selected model: Linear Regression with StandardScaler")
print("This model achieved:")
print("- Rougher sMAPE: 11.9253")
print("- Final sMAPE: 9.7461")

Selected model: Linear Regression with StandardScaler
This model achieved:
- Rougher sMAPE: 11.9253
- Final sMAPE: 9.7461


In [12]:
# Evaluate final model performance using sMAPE
# Extract true target values from full dataset using test index
true_rougher = full.loc[test.index, 'rougher.output.recovery']
true_final = full.loc[test.index, 'final.output.recovery']

# Calculate final sMAPE score
final_score = final_smape(true_rougher, rougher_pred, true_final, final_pred)
print(f'Final sMAPE Score: {final_score:.4f}')

Final sMAPE Score: 10.3727


## 8. Conclusion

This project successfully built and validated a machine learning pipeline to predict gold and final output recovery rates from ore flotation sensor data.

### Key Findings

**Data quality:** Recovery calculations matched the provided formula exactly (MAE = 0.0000), validating the dataset before any modeling work. Feed particle size distributions were closely aligned between train and test sets (~2.3% difference in means), confirming reliable model evaluation conditions.

**EDA insights:** Metal concentration histograms confirmed the purification process works as expected — gold enrichment is the strongest and most consistent, followed by silver, with lead showing the most variability. A consistent ~1% outlier rate across all processing stages suggests these extreme values may reflect real operational events rather than data errors.

**Model selection:** Three models were evaluated using 5-fold cross-validation with sMAPE scoring:

| Model | Avg sMAPE |
|---|---|
| **Linear Regression** | **10.84** |
| Random Forest | 11.74 |
| Decision Tree | 19.08 |

**Linear Regression** was selected as the best model. Its superior performance over ensemble methods suggests the dominant relationships between sensor features and recovery rates are approximately linear — consistent with the physics-based nature of flotation chemistry.

**Final test performance:** The model achieved a **final sMAPE of 10.3727** on the held-out test set, consistent with cross-validation results, demonstrating strong generalization to unseen data. This level of accuracy is suitable for supporting operational decisions in gold recovery optimization.